# Step 09 — Build per-workflow ground_truth candidates with smell features

> For every retained workflow, compute the per-file feature vector that the Step 10 stratified sampler will need: per-smell `instance_count`, `total_unique_smells`, and `workflow_run_count`. Smell counts come from running the GHA static analyzer (Khatami et al.) on each YAML file.

**Paper section:** Step 3 — Ground-truth construction / Stratified sampling (candidate-pool construction).

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/gha_smells.csv` | CSV (~2 GB decompressed; ship as `gha_smells.csv.xz`, decompress before running) | External: GHA analyzer (Khatami et al.) run over `data/downloaded_workflows/` |
| `data/8_dataset_enriched.csv` | CSV | Step 8 |
| `data/downloaded_workflows/` | Directory of YAML workflow files (~770 MB, ~145 k files) | Step 8 (mined corpus); auto-extracted from `downloaded_workflows.tar.xz` |
| `data/gha_analyzer_exclusions.txt` | Text, one `file_key` per line | The workflows that GHA cannot parse; Step 09 leaves them out, and stops if `gha_smells.csv` misses any other workflow |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/9_ground_truth_candidates.csv` | CSV (one row per workflow file with smell features) | Step 10 (sampling) |
| `data/step9_report.json` | JSON | Step 10 |

## Manual intervention
Decompress `data/gha_smells.csv.xz` first: `unxz -k data/gha_smells.csv.xz`.



In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 09 — starting: build ground_truth candidates from gha_smells.csv (large file, may take minutes)...", flush=True)
"""Step 09: Build ground_truth candidates from GHA smells and enriched dataset.

Cross-references GHA static analysis results (data/gha_smells.csv) with the
enriched dataset (data/8_dataset_enriched.csv) and the YAML cache
(data/downloaded_workflows/) to produce 9_ground_truth_candidates.csv — a per-file
smell profile with quality metadata used by Step 10 for stratified ground_truth
sampling.

Inputs:
  data/gha_smells.csv            GHA detection results (file, smell, description, line, code)
    data/8_dataset_enriched.csv    Enriched dataset (stars, run_count, language, …)
  data/downloaded_workflows/       Cached YAML workflow files (owner@repo@filename.yml)
  data/gha_analyzer_exclusions.txt Workflows the GHA analyzer cannot parse (left out)

Outputs:
  data/9_ground_truth_candidates.csv          Per-file smell profile + quality metadata
  data/step9_report.json  Summary statistics and per-smell coverage
"""

import argparse
import csv
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────────
# Smell catalog — 22 smells from Khatami et al. SCAM 2024, Table III.
#
# SMELL_CATALOG is keyed by the short PascalCase name that gha_smells.csv
# carries; its value is (paper_id, family, latex), paper_id being the Khatami
# Table III ordinal (1..22).
# ──────────────────────────────────────────────────────────────────────────────
from dataset_utils.ground_truth_sampling import SMELL_CATALOG

PAPER_ID_TO_SHORT: dict[int, str] = {
    paper_id: short for short, (paper_id, _family, _latex) in SMELL_CATALOG.items()
}

# Maximum occurrences stored per smell per file (keeps smells_json bounded)
MAX_OCCURRENCES_PER_SMELL = 10

# Some Formatting rows in gha_smells.csv carry raw lint passthrough in the
# format "<line>:<col>: <human message> (<rule>)" instead of a original
# PascalCase identifier — the GHA detector appears to bubble up its
# underlying linter's output verbatim for sub-rules outside the catalog.
# Only `indentation` maps to a Khatami original smell (CorrectIndentation,
# id 14). Other rules (brackets, trailing-spaces, comments, ...) have no
# paper equivalent and are dropped.
RAW_LINT_DESC_RE = re.compile(r"^\d+:\d+:.*\(([a-z\-]+)\)\s*$")
RAW_LINT_RULE_MAP: dict[str, str] = {
    "indentation": "CorrectIndentation",
}

# ──────────────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────────────

def build_file_key(repository: str, file_path: str) -> str:
    """Build cache file key from enriched dataset columns.

    Maps "owner/repo" + ".github/workflows/ci.yml"
      → "owner@repo@ci.yml"
    which matches the naming convention of data/downloaded_workflows/.
    """
    repo_part = repository.replace("/", "@")
    filename = file_path.split("/")[-1]
    return f"{repo_part}@{filename}"

def count_yaml_lines(yaml_path: Path) -> int:
    """Return line count of a YAML file, 0 if unreadable."""
    try:
        with yaml_path.open("r", encoding="utf-8", errors="replace") as f:
            return sum(1 for _ in f)
    except OSError:
        return 0

# ──────────────────────────────────────────────────────────────────────────────
# Loaders
# ──────────────────────────────────────────────────────────────────────────────

def load_enriched_meta(path: Path) -> dict[str, dict]:
    """Load enriched dataset; index by file_key.

    Returns: {file_key → {repository, file_path, repo_language, repo_stars,
                           workflow_run_count, repo_commits}}
    """
    meta: dict[str, dict] = {}
    csv.field_size_limit(10**7)
    with path.open(encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            key = build_file_key(row["repository"], row["file_path"])
            meta[key] = {
                "repository": row["repository"],
                "file_path": row["file_path"],
                "repo_language": row.get("repo_language", ""),
                "repo_stars": int(row.get("repo_stars") or 0),
                "workflow_run_count": int(row.get("workflow_run_count") or 0),
                "repo_commits": int(row.get("repo_commits") or 0),
            }
    return meta

def load_smell_profiles(
    smells_path: Path,
    enriched_keys: set[str],
) -> dict[str, dict]:
    """Build per-file smell profile from gha_smells.csv.

    Description column resolution (in order):
      1. PascalCase key in SMELL_CATALOG  → use directly
      2. raw lint format with rule in MAP → use mapped catalog key
      3. raw lint with unmapped rule       → skip (out of paper scope)
      4. anything else                     → skip and count for visibility

    Returns:
      {file_key → {
        "total_instances": int,
        "smells": {
          pascal → {
            "paper_id": int,
            "instance_count": int,
            "occurrences": [{"line": str, "code": str}, ...]
          }
        }
      }}
    """
    profiles: dict[str, dict] = {}
    csv.field_size_limit(10**7)
    skipped: Counter = Counter()

    with smells_path.open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            file_key = row["file"]
            if file_key not in enriched_keys:
                continue

            description = row.get("description", "")

            if description in SMELL_CATALOG:
                pascal = description
            else:
                m = RAW_LINT_DESC_RE.match(description)
                if m is None:
                    skipped["non_original"] += 1
                    continue
                pascal = RAW_LINT_RULE_MAP.get(m.group(1))
                if pascal is None:
                    skipped[f"raw_lint:{m.group(1)}"] += 1
                    continue

            paper_id, _, _ = SMELL_CATALOG[pascal]
            pascal = PAPER_ID_TO_SHORT[paper_id]  # long tool name -> short original (Step 10+ taxonomy)

            if file_key not in profiles:
                profiles[file_key] = {"total_instances": 0, "smells": {}}

            profile = profiles[file_key]
            profile["total_instances"] += 1

            smells = profile["smells"]
            if pascal not in smells:
                smells[pascal] = {
                    "paper_id": paper_id,
                    "instance_count": 0,
                    "occurrences": [],
                }

            entry = smells[pascal]
            entry["instance_count"] += 1
            if len(entry["occurrences"]) < MAX_OCCURRENCES_PER_SMELL:
                entry["occurrences"].append({
                    "line": row.get("line", ""),
                    "code": row.get("code", "")[:300],
                })

    if skipped:
        total = sum(skipped.values())
        print(f"  Skipped {total:,} non-catalog rows:")
        for reason, n in skipped.most_common():
            print(f"    {n:>9,}  {reason}")

    return profiles

# ──────────────────────────────────────────────────────────────────────────────
# Candidate builder
# ──────────────────────────────────────────────────────────────────────────────

def build_candidates(
    enriched_meta: dict[str, dict],
    smell_profiles: dict[str, dict],
    yaml_dir: Path,
) -> list[dict]:
    """Merge enriched metadata with smell profiles into candidate rows.

    Each row represents one workflow file.  smells_json encodes all catalog
    smells detected in that file, with occurrence details for step 11 and
    manual review.
    """
    candidates = []

    for file_key, meta in enriched_meta.items():
        yaml_path = yaml_dir / file_key
        profile = smell_profiles.get(file_key, {"total_instances": 0, "smells": {}})
        smells = profile["smells"]

        smells_list = sorted(
            [
                {
                    "pascal": pascal,
                    "paper_id": entry["paper_id"],
                    "instance_count": entry["instance_count"],
                    "occurrences": entry["occurrences"],
                }
                for pascal, entry in smells.items()
            ],
            key=lambda s: s["paper_id"],
        )

        candidates.append({
            "file_key": file_key,
            "repository": meta["repository"],
            "file_path": meta["file_path"],
            # Relative path keeps the CSV reproducible across machines.
            "yaml_path": str(yaml_path),
            "file_line_count": count_yaml_lines(yaml_path),
            "repo_stars": meta["repo_stars"],
            "workflow_run_count": meta["workflow_run_count"],
            "repo_commits": meta["repo_commits"],
            "repo_language": meta["repo_language"],
            "total_unique_smells": len(smells),
            "total_smell_instances": profile["total_instances"],
            "smells_json": json.dumps(smells_list, ensure_ascii=False),
        })

    return candidates

# ──────────────────────────────────────────────────────────────────────────────
# Writers
# ──────────────────────────────────────────────────────────────────────────────

_OUTPUT_FIELDS = [
    "file_key",
    "repository",
    "file_path",
    "yaml_path",
    "file_line_count",
    "repo_stars",
    "workflow_run_count",
    "repo_commits",
    "repo_language",
    "total_unique_smells",
    "total_smell_instances",
    "smells_json",
]

def write_csv(candidates: list[dict], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=_OUTPUT_FIELDS)
        writer.writeheader()
        writer.writerows(candidates)

def write_report(
    candidates: list[dict],
    report_path: Path,
    output_path: Path | None = None,
    *,
    enriched_sha256: str | None = None,
    gha_smells_sha256: str | None = None,
    prior_report: Path | None = None,
    analyzer_excluded_count: int = 0,
) -> None:
    """Write summary statistics and per-smell coverage to JSON."""
    total = len(candidates)
    with_smells = sum(1 for c in candidates if c["total_unique_smells"] > 0)

    smell_coverage: Counter = Counter()
    smell_solo_coverage: Counter = Counter()  # files where this is the only smell
    for c in candidates:
        smells = json.loads(c["smells_json"])
        n = len(smells)
        for s in smells:
            pascal = s["pascal"]
            smell_coverage[pascal] += 1
            if n == 1:
                smell_solo_coverage[pascal] += 1

    dist: Counter = Counter(c["total_unique_smells"] for c in candidates)

    # Cumulative distribution
    cumul_dist = {}
    running = 0
    for k in sorted(dist):
        running += dist[k]
        cumul_dist[str(k)] = {
            "count": dist[k],
            "cumulative": running,
            "cumulative_pct": round(running / total * 100, 1),
        }

    report = {
        "total_candidates": total,
        "files_with_catalog_smells": with_smells,
        "clean_files_no_catalog_smells": total - with_smells,
        "catalog_entries": len(SMELL_CATALOG),
        "coverage_per_smell": {
            desc: {
                "files_with_smell": smell_coverage[desc],
                "solo_files": smell_solo_coverage.get(desc, 0),
            }
            for desc in sorted(smell_coverage, key=lambda d: -smell_coverage[d])
        },
        "distribution_total_unique_smells": cumul_dist,
    }

    if enriched_sha256 is not None:
        report["enriched_sha256"] = enriched_sha256
    if gha_smells_sha256 is not None:
        report["gha_smells_sha256"] = gha_smells_sha256
    if prior_report is not None:
        report["prior_report"] = str(prior_report)
    if analyzer_excluded_count:
        report["analyzer_excluded_count"] = analyzer_excluded_count
    if output_path is not None and output_path.exists():
        import sys as _sys
        try:
            _sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
        except NameError:
            pass  # Jupyter: __file__ undefined; dataset_utils on path via wheel install
        from dataset_utils.hashing import compute_file_hash
        report["output_file"] = str(output_path)
        report["output_sha256"] = compute_file_hash(output_path)

    report_path.parent.mkdir(parents=True, exist_ok=True)
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

# ──────────────────────────────────────────────────────────────────────────────
# CLI
# ──────────────────────────────────────────────────────────────────────────────

def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Build ground_truth candidates dataset from GHA smells and enriched metadata."
    )
    p.add_argument(
        "--smells",
        type=Path,
        default=Path("data/gha_smells.csv"),
        help="GHA smells CSV (default: data/gha_smells.csv)",
    )
    p.add_argument(
        "--enriched",
        type=Path,
        default=Path("data/8_dataset_enriched.csv"),
        help="Enriched dataset CSV (default: data/8_dataset_enriched.csv)",
    )
    p.add_argument(
        "--yaml-cache",
        type=Path,
        default=Path("data/downloaded_workflows"),
        help="Directory with cached YAML files (default: data/downloaded_workflows)",
    )
    p.add_argument(
        "--output",
        type=Path,
        default=Path("data/9_ground_truth_candidates.csv"),
        help="Output candidates CSV (default: data/9_ground_truth_candidates.csv)",
    )
    p.add_argument(
        "--report",
        type=Path,
        default=Path("data/step9_report.json"),
        help="Output report JSON (default: data/step9_report.json)",
    )
    p.add_argument(
        "--prior-report",
        type=Path,
        default=Path("data/step8_report.json"),
        help="Step 8 report JSON used for SHA-chain integrity check (default: data/step8_report.json)",
    )
    return p.parse_args()

def main() -> None:
    args = parse_args()

    import sys as _sys
    try:
        _sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
    except NameError:
        pass  # Jupyter: __file__ undefined; dataset_utils on path via wheel install
    from dataset_utils.integrity import (
        PipelineIntegrityError,
        assert_input_matches_report,
        hash_source_file,
    )

    if not args.prior_report.exists():
        msg = (
            f"step 09 enriched input: prior report not found at {args.prior_report}. "
            "Run step 8 first."
        )
        raise PipelineIntegrityError(msg)
    prior_report = args.prior_report

    enriched_sha = assert_input_matches_report(
        input_path=args.enriched,
        prior_report_path=prior_report,
        label="step 09 enriched input",
    )
    # gha_smells.csv is produced by an external tool (separate repo);
    # we record its SHA for audit but do not assert against a prior step.
    gha_smells_sha = hash_source_file(args.smells, "step 09 gha_smells input")
    print(f"  enriched SHA  : {enriched_sha[:16]}…")
    print(f"  gha_smells SHA: {gha_smells_sha[:16]}…")

    print("Loading enriched dataset...")
    from dataset_utils.bootstrap import ensure_extracted; ensure_extracted(args.yaml_cache)
    enriched_meta = load_enriched_meta(args.enriched)

    # F6: fail-fast if gha_smells.csv is missing any file_key from the current
    # enriched dataset. Historical bug: a stale gha_smells (from a previous
    # dataset_enriched) silently classified the missing workflows as "clean"
    # (0 smells), biasing RQ1 recall and ground_truth sampling downstream.
    #
    # Exceptions: files that the GHA analyzer cannot parse (e.g., bare-boolean
    # `if` conditions that fail its non-string contract validation) are listed
    # in data/gha_analyzer_exclusions.txt. Those are excluded from F6 coverage
    # and from ground_truth candidates (recorded as "analyzer_excluded" in report).
    import csv as _csv
    import sys as __sys
    try:
        __sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
    except NameError:
        pass  # Jupyter: __file__ undefined; dataset_utils on path via wheel install
    _csv.field_size_limit(__sys.maxsize)
    gha_file_keys: set[str] = set()
    with args.smells.open("r", encoding="utf-8", newline="") as _sf:
        for _r in _csv.DictReader(_sf):
            gha_file_keys.add(_r["file"])

    exclusions_path = Path("data/gha_analyzer_exclusions.txt")
    analyzer_excluded: set[str] = set()
    if exclusions_path.exists():
        analyzer_excluded = {
            line.strip() for line in exclusions_path.read_text().splitlines() if line.strip()
        }

    enriched_keys = set(enriched_meta.keys())
    # A file is "covered" if it appears in gha_smells (has ≥1 smell) OR is
    # listed as analyzer-excluded. Missing = enriched but neither.
    missing = enriched_keys - gha_file_keys - analyzer_excluded
    if missing:
        sample = sorted(missing)[:10]
        msg = (
            f"step 09: gha_smells.csv ({args.smells}) is missing {len(missing)} "
            f"of {len(enriched_keys)} file_keys present in {args.enriched} "
            f"(and {len(analyzer_excluded)} known analyzer exclusions do not cover them). "
            f"Running step 10 with this input would silently classify the "
            f"missing workflows as 'clean' (0 smells). "
            f"Run the GHA analyzer on the current yaml cache to regenerate "
            f"gha_smells.csv before proceeding. First missing: {sample}"
        )
        raise PipelineIntegrityError(msg)

    # Drop analyzer-excluded files from the enriched universe for step 10 —
    # they cannot produce valid smell profiles, so including them as "0 smells"
    # would bias RQ1 recall.
    if analyzer_excluded:
        before = len(enriched_meta)
        enriched_meta = {k: v for k, v in enriched_meta.items() if k not in analyzer_excluded}
        print(
            f"  analyzer exclusions applied: {before - len(enriched_meta)} files removed "
            f"(see {exclusions_path})"
        )
    enriched_keys = set(enriched_meta.keys())
    print(f"  {len(enriched_meta):,} files")

    print("Processing GHA smells (large file — may take a moment)...")
    smell_profiles = load_smell_profiles(args.smells, enriched_keys)
    covered = len(smell_profiles)
    clean = len(enriched_meta) - covered
    print(f"  {covered:,} files with >= 1 catalog smell")
    print(f"  {clean:,} files with no catalog smells (will be included as clean)")

    print("Building candidate rows + counting YAML lines...")
    candidates = build_candidates(enriched_meta, smell_profiles, args.yaml_cache)

    print(f"Writing {args.output} ...")
    write_csv(candidates, args.output)

    print(f"Writing {args.report} ...")
    write_report(
        candidates,
        args.report,
        args.output,
        enriched_sha256=enriched_sha,
        gha_smells_sha256=gha_smells_sha,
        prior_report=prior_report,
        analyzer_excluded_count=len(analyzer_excluded),
    )

    # ── Console summary ───────────────────────────────────────────────────────
    print()
    print("=" * 60)
    print("Ground truth Candidates — Summary")
    print("=" * 60)
    print(f"  Total candidates    : {len(candidates):,}")
    print(f"  With catalog smells : {covered:,}")
    print(f"  Clean files         : {clean:,}")
    print(f"  Catalog entries     : {len(SMELL_CATALOG)}")

    # Coverage per smell
    smell_cov: Counter = Counter()
    for c in candidates:
        for s in json.loads(c["smells_json"]):
            smell_cov[s["pascal"]] += 1
    print()
    print("  Coverage per Khatami smell (files):")
    # smell_cov keys are SHORT original pascals; resolve via the SHORT-keyed
    # original catalog, mapping paper_id back to the gha tool category label.
    _pid_to_gha = {pid: gha for gha, _sec, pid in SMELL_CATALOG.values()}
    for pascal, n in smell_cov.most_common():
        paper_id, _family, _ = SMELL_CATALOG[pascal]
        gha_cat = _pid_to_gha.get(paper_id, "?")
        print(f"    [{gha_cat:20s}] id={paper_id:2}  {n:6,}  {pascal}")

    # Threshold summary
    dist: Counter = Counter(c["total_unique_smells"] for c in candidates)
    total = len(candidates)
    print()
    print("  Files by total_unique_smells threshold:")
    cumul = 0
    for k in sorted(dist):
        if k > 15:
            break
        cumul += dist[k]
        print(f"    <= {k:2d} smells: {cumul:6,} files ({cumul/total*100:.1f}%)")

main()


Step 09 — starting: build ground_truth candidates from gha_smells.csv (large file, may take minutes)...


  enriched SHA  : 49e005037a23b346…
  gha_smells SHA: 69760390f69291fb…
Loading enriched dataset...
[bootstrap] data/downloaded_workflows already present — skipping extract.


  analyzer exclusions applied: 7 files removed (see data/gha_analyzer_exclusions.txt)
  49,829 files
Processing GHA smells (large file — may take a moment)...


  Skipped 176,882 non-catalog rows:
       82,599  raw_lint:comments
       39,080  raw_lint:brackets
       21,380  raw_lint:trailing-spaces
       12,084  raw_lint:braces
        4,167  raw_lint:commas
        3,963  raw_lint:colons
        3,542  raw_lint:comments-indentation
        3,124  raw_lint:truthy
        2,530  raw_lint:new-line-at-end-of-file
        1,802  raw_lint:empty-lines
        1,640  raw_lint:hyphens
          971  raw_lint:line-length
  49,829 files with >= 1 catalog smell
  0 files with no catalog smells (will be included as clean)
Building candidate rows + counting YAML lines...


Writing data/9_ground_truth_candidates.csv ...


Writing data/step9_report.json ...



Ground truth Candidates — Summary
  Total candidates    : 49,829
  With catalog smells : 49,829
  Clean files         : 0
  Catalog entries     : 22



  Coverage per Khatami smell (files):
    [?                   ] id=12  47,825  UnsafeDeployOnFork
    [?                   ] id= 5  43,375  MissingJobTimeout
    [?                   ] id=15  36,049  MutableRunnerVersion
    [?                   ] id= 2  36,031  UnpinnedActionVersion
    [?                   ] id=14  33,336  CorrectIndentation
    [?                   ] id= 1  29,490  MissingPermExternalActions
    [?                   ] id=16  28,870  NameRunSteps
    [?                   ] id= 8  27,765  MissingPathFilters
    [?                   ] id= 6  27,006  MissingConcurrencyPR
    [?                   ] id= 7  26,826  MissingConcurrencyBranch
    [?                   ] id=18  24,937  MultiCommandStep
    [?                   ] id=19  21,161  SingleOSExecution
    [?                   ] id=21  15,432  AddCommentsToWorkflows
    [?                   ] id= 4   8,370  MissingForkGuardPR
    [?                   ] id=20   8,362  UnpinnedDependencies
    [?                   ] id

---

**Next step:** [`09b_compute_loc_cap.ipynb`](./09b_compute_loc_cap.ipynb)
